# 31 — Incremental RAG

> **Tier 8 | Incremental / Production**

## The Problem

Every previous notebook processes the entire document from scratch on each run.
In production, documents change: a policy PDF gets a new section, a report page is
updated, a chapter is deleted. Re-embedding the entire document on every change is:
- Expensive (API cost × chunk count)
- Slow (embedding 100+ chunks takes 15–20 seconds)
- Unnecessary (most pages are identical)

## Solution: Content-Addressed Incremental Indexing

Hash every page and every chunk at ingest time. Store the hash as the Qdrant point ID.
On document update:
1. Re-extract page text, re-hash each page
2. **Diff** new page hashes against the stored manifest
3. Process **only** changed/added pages
4. Delete vectors for removed/changed pages
5. Embed and upsert only new/changed chunks

Unchanged pages → zero embed calls, zero upsert operations.

## Edge Cases Handled

| Scenario | Handling |
|---|---|
| Page content changed | Delete old chunks, embed+upsert new chunks |
| Page added (doc grows) | Embed+upsert new page chunks only |
| Page deleted (doc shrinks) | Delete all chunks for that page |
| Chunk text identical but page number changed | Detected via `doc_id::page::text` hash — treated as new |
| Whole document replaced | Full diff → all old deleted, all new added |
| Same document re-ingested | All hashes match → zero operations |
| Manifest missing (first ingest) | Treated as full ingest, manifest created |
| Qdrant point ID collision | Hash-as-ID makes upsert idempotent — safe to repeat |
| Empty page added/removed | Zero-text pages hashed and tracked correctly |
| Chunk split boundary shifts (overlap change) | Config hash in manifest detects chunking param change → full re-index |

## PDF Framework: `pymupdf get_text("text")` per page

Per-page extraction is the natural unit for incremental processing —
each page becomes an independently hashable, diffable unit.


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INGEST ["🔵  FIRST INGEST"]
        PDF1(["📄 doc_v1.pdf"])
        PDF1 --> EXTRACT1["Extract per-page text\n(pymupdf get_text)"]
        EXTRACT1 --> HASH1["Hash each page\nSHA-256[:16]"]
        HASH1 --> MANIFEST1[("manifest.json\npage → hash")]
        EXTRACT1 --> CHUNK1["Chunk pages"]
        CHUNK1 --> IDHASH["chunk_hash = SHA-256\n(doc_id::page::text)"]
        IDHASH --> EMB1["Embed all chunks"]
        EMB1 --> QDRANT[("Qdrant\nhash as point ID")]
    end
    subgraph UPDATE ["🟢  INCREMENTAL UPDATE"]
        PDF2(["📄 doc_v2.pdf"])
        PDF2 --> EXTRACT2["Extract per-page text"]
        EXTRACT2 --> HASH2["Hash each page"]
        HASH2 --> DIFF{"Diff vs\nmanifest.json"]
        DIFF -->|"unchanged"| SKIP(["⏭ Skip — zero API calls"])
        DIFF -->|"changed"| DEL["Delete old chunk IDs\nfrom Qdrant"]
        DEL --> RECHUNK["Re-chunk changed pages"]
        RECHUNK --> EMBNEW["Embed only new chunks"]
        EMBNEW --> UPSERT["Upsert new vectors"]
        DIFF -->|"added"| RECHUNK
        DIFF -->|"deleted"| DEL2["Delete all chunk IDs\nfor removed page"]
        UPSERT --> MANIFEST2[("manifest.json\nupdated")]
    end
    style INGEST fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style UPDATE fill:#dcfce7,stroke:#16a34a,color:#14532d
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "pymupdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, hashlib, copy, shutil
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Set
from dotenv import load_dotenv

import boto3
import fitz   # pymupdf
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, PayloadSchemaType
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")
print(f"pymupdf: {fitz.__version__}")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME = "incremental_rag_31"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 5

PDF_PATH     = r"C:\Users\Administrator\RAG\data\medicaid.pdf"
DOC_ID       = "medicaid"                        # logical document identifier
MANIFEST_DIR = r"C:\Users\Administrator\RAG\data\manifests"

Path(MANIFEST_DIR).mkdir(parents=True, exist_ok=True)

print(f"Collection   : {COLLECTION_NAME}")
print(f"Doc ID       : {DOC_ID}")
print(f"Manifest dir : {MANIFEST_DIR}")
print(f"PDF          : {PDF_PATH}  exists={os.path.exists(PDF_PATH)}")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
# Payload indexes for filtered queries
qdrant.create_payload_index(COLLECTION_NAME, "doc_id",  PayloadSchemaType.KEYWORD)
qdrant.create_payload_index(COLLECTION_NAME, "page",    PayloadSchemaType.INTEGER)
qdrant.create_payload_index(COLLECTION_NAME, "chunk_id",PayloadSchemaType.KEYWORD)
print(f'Created "{COLLECTION_NAME}" with payload indexes')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out


def call_llm(system: str, user_content: str, max_tokens: int = 512) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]


test_emb = embed_text("incremental rag test")
print(f"Embed OK — dim={len(test_emb)}")


## Step 5 — Core Primitives: Hashing & Chunking

### Chunk ID design
```
chunk_id = SHA-256(doc_id + "::" + str(page) + "::" + text.strip())
```
Using the hash **as the Qdrant point ID** makes every upsert idempotent — re-ingesting
an identical chunk is a no-op. Changing any of `doc_id`, `page`, or `text` produces a
different ID, so the old vector is correctly superseded.

### Config hash in manifest
If `CHUNK_SIZE` or `CHUNK_OVERLAP` changes, all chunk boundaries shift. The manifest
stores a `config_hash` — if it mismatches on load, a full re-index is triggered automatically.


In [ ]:
def chunk_id(doc_id: str, page: int, text: str) -> str:
    key = f"{doc_id}::{page}::{text.strip()}"
    h = hashlib.sha256(key.encode('utf-8', errors='replace')).hexdigest()
    return str(uuid.UUID(h[:32]))   # Qdrant requires UUID or uint — derive UUID from hash


def page_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8', errors='replace')).hexdigest()


def config_hash() -> str:
    cfg = f"chunk_size={CHUNK_SIZE},overlap={CHUNK_OVERLAP},model={EMBEDDING_MODEL}"
    return hashlib.sha256(cfg.encode()).hexdigest()[:16]


def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def extract_pages(pdf_path: str) -> Dict[int, str]:
    """Return {page_number: page_text} for every page in the PDF."""
    doc = fitz.open(pdf_path)
    pages = {}
    for i in range(len(doc)):
        text = ' '.join(doc[i].get_text('text').split()).strip()
        pages[i + 1] = text   # 1-indexed
    doc.close()
    return pages


def page_to_chunks(doc_id: str, page: int, text: str) -> List[Dict]:
    """Split page text into chunks, each with a deterministic chunk_id."""
    chunks = []
    for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
        cid = chunk_id(doc_id, page, sub)
        chunks.append({
            'chunk_id': cid,
            'text'    : sub,
            'doc_id'  : doc_id,
            'page'    : page,
        })
    return chunks


print("Hashing and chunking primitives defined.")
print(f"Config hash: {config_hash()}")


## Step 6 — Manifest: Store & Load

The manifest is a JSON sidecar file stored alongside the document:
```json
{
  "doc_id": "medicaid",
  "config_hash": "a1b2c3d4",
  "pages": {
    "1": {"page_hash": "abc123", "chunk_ids": ["id1", "id2"]},
    "2": {"page_hash": "def456", "chunk_ids": ["id3"]},
    ...
  }
}
```


In [ ]:
def manifest_path(doc_id: str) -> str:
    return os.path.join(MANIFEST_DIR, f"{doc_id}_manifest.json")


def load_manifest(doc_id: str) -> Optional[Dict]:
    """Load existing manifest. Returns None if not found."""
    path = manifest_path(doc_id)
    if not os.path.exists(path):
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_manifest(doc_id: str, manifest: Dict):
    path = manifest_path(doc_id)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest saved: {path}")


def build_manifest(doc_id: str, pages: Dict[int, str],
                   page_chunks: Dict[int, List[Dict]]) -> Dict:
    """Build a fresh manifest from extracted pages and their chunks."""
    return {
        'doc_id'     : doc_id,
        'config_hash': config_hash(),
        'n_pages'    : len(pages),
        'pages'      : {
            str(p): {
                'page_hash': page_hash(text),
                'chunk_ids': [c['chunk_id'] for c in page_chunks.get(p, [])],
                'n_chunks' : len(page_chunks.get(p, [])),
            }
            for p, text in pages.items()
        }
    }


print("Manifest helpers defined.")


## Step 7 — Diff Engine

Compares old manifest against new page hashes and returns exactly what needs to change.


In [ ]:
def diff_manifests(old_manifest: Optional[Dict],
                   new_pages: Dict[int, str]) -> Dict:
    """
    Compare old manifest against freshly extracted page hashes.

    Returns:
      changed_pages  : pages whose text changed
      added_pages    : pages that are new (doc grew)
      deleted_pages  : pages that no longer exist (doc shrank)
      unchanged_pages: pages identical to the stored manifest
      chunk_ids_to_delete: all Qdrant point IDs that must be removed
      force_full_reindex : True if config changed or manifest missing
    """
    result = {
        'changed_pages'       : set(),
        'added_pages'         : set(),
        'deleted_pages'       : set(),
        'unchanged_pages'     : set(),
        'chunk_ids_to_delete' : set(),
        'force_full_reindex'  : False,
    }

    # ── Edge case: no manifest (first ingest) ────────────────────────────────
    if old_manifest is None:
        result['added_pages']       = set(new_pages.keys())
        result['force_full_reindex']= True
        return result

    # ── Edge case: config changed → full re-index ────────────────────────────
    if old_manifest.get('config_hash') != config_hash():
        print("  [WARN] Chunking config changed — forcing full re-index")
        result['added_pages']        = set(new_pages.keys())
        result['chunk_ids_to_delete']= {
            cid
            for pdata in old_manifest['pages'].values()
            for cid in pdata['chunk_ids']
        }
        result['force_full_reindex'] = True
        return result

    old_pages_str = old_manifest['pages']
    old_page_nums = {int(p) for p in old_pages_str}
    new_page_nums = set(new_pages.keys())

    # ── Deleted pages ────────────────────────────────────────────────────────
    for p in old_page_nums - new_page_nums:
        result['deleted_pages'].add(p)
        for cid in old_pages_str[str(p)]['chunk_ids']:
            result['chunk_ids_to_delete'].add(cid)

    # ── Changed vs unchanged pages ───────────────────────────────────────────
    for p in old_page_nums & new_page_nums:
        old_hash = old_pages_str[str(p)]['page_hash']
        new_h    = page_hash(new_pages[p])
        if old_hash != new_h:
            result['changed_pages'].add(p)
            # delete ALL old chunk IDs for this page
            for cid in old_pages_str[str(p)]['chunk_ids']:
                result['chunk_ids_to_delete'].add(cid)
        else:
            result['unchanged_pages'].add(p)

    # ── Added pages ──────────────────────────────────────────────────────────
    result['added_pages'] = new_page_nums - old_page_nums

    return result


print("Diff engine defined.")


## Step 8 — Incremental Ingest Engine

Orchestrates the full incremental pipeline: diff → delete → embed → upsert → update manifest.


In [ ]:
def delete_chunks_from_qdrant(chunk_ids: Set[str]):
    """Delete a set of chunk IDs from Qdrant in one batch call."""
    if not chunk_ids:
        return
    qdrant.delete(
        collection_name=COLLECTION_NAME,
        points_selector=list(chunk_ids),
    )


def upsert_chunks(chunks: List[Dict]):
    """Embed and upsert a list of chunk dicts into Qdrant."""
    if not chunks:
        return
    texts = [c['text'] for c in chunks]
    embs  = embed_batch(texts, label='[embed]')
    pts   = [
        PointStruct(
            id     = chunks[i]['chunk_id'],   # deterministic hash ID
            vector = embs[i],
            payload= {
                'text'    : chunks[i]['text'],
                'doc_id'  : chunks[i]['doc_id'],
                'page'    : chunks[i]['page'],
                'chunk_id': chunks[i]['chunk_id'],
            }
        )
        for i in range(len(chunks))
    ]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)


def incremental_ingest(pdf_path: str, doc_id: str,
                       verbose: bool = True) -> Dict:
    """
    Main entry point. Runs a full incremental ingest:
      1. Extract page texts
      2. Load existing manifest
      3. Diff new pages vs manifest
      4. Delete stale vectors
      5. Embed + upsert only new/changed chunks
      6. Save updated manifest

    Returns a stats dict describing what was done.
    """
    t0 = time.time()

    # 1. Extract
    pages = extract_pages(pdf_path)

    # 2. Load manifest
    old_manifest = load_manifest(doc_id)
    is_first     = old_manifest is None

    # 3. Diff
    diff = diff_manifests(old_manifest, pages)

    pages_to_process = diff['added_pages'] | diff['changed_pages']
    pages_to_skip    = diff['unchanged_pages']

    if verbose:
        print(f"\nIngest: doc_id='{doc_id}'  pdf={os.path.basename(pdf_path)}")
        print(f"  Total pages     : {len(pages)}")
        print(f"  Unchanged       : {len(pages_to_skip)}  (skip)")
        print(f"  Changed         : {sorted(diff['changed_pages'])}")
        print(f"  Added           : {sorted(diff['added_pages'])}")
        print(f"  Deleted         : {sorted(diff['deleted_pages'])}")
        print(f"  Vectors to delete: {len(diff['chunk_ids_to_delete'])}")

    # 4. Delete stale vectors
    t_del = time.time()
    delete_chunks_from_qdrant(diff['chunk_ids_to_delete'])
    t_del = (time.time() - t_del) * 1000

    # 5. Chunk + embed + upsert only affected pages
    new_chunks_by_page: Dict[int, List[Dict]] = {}
    all_new_chunks: List[Dict] = []

    for p in pages_to_process:
        c = page_to_chunks(doc_id, p, pages[p])
        new_chunks_by_page[p] = c
        all_new_chunks.extend(c)

    # Keep existing chunk records for unchanged pages
    unchanged_chunks_by_page: Dict[int, List[Dict]] = {}
    if old_manifest and not diff['force_full_reindex']:
        for p in pages_to_skip:
            pdata = old_manifest['pages'].get(str(p), {})
            unchanged_chunks_by_page[p] = [
                {'chunk_id': cid} for cid in pdata.get('chunk_ids', [])
            ]

    t_emb = time.time()
    upsert_chunks(all_new_chunks)
    t_emb = (time.time() - t_emb) * 1000

    # 6. Build and save updated manifest
    all_page_chunks = {**unchanged_chunks_by_page, **new_chunks_by_page}
    new_manifest    = build_manifest(doc_id, pages, all_page_chunks)
    save_manifest(doc_id, new_manifest)

    t_total = (time.time() - t0) * 1000
    n_vec   = qdrant.get_collection(COLLECTION_NAME).points_count

    stats = {
        'is_first_ingest': is_first,
        'pages_total'    : len(pages),
        'pages_unchanged': len(pages_to_skip),
        'pages_changed'  : len(diff['changed_pages']),
        'pages_added'    : len(diff['added_pages']),
        'pages_deleted'  : len(diff['deleted_pages']),
        'chunks_deleted' : len(diff['chunk_ids_to_delete']),
        'chunks_added'   : len(all_new_chunks),
        'embed_calls'    : len(all_new_chunks),
        'vectors_total'  : n_vec,
        'delete_ms'      : t_del,
        'embed_ms'       : t_emb,
        'total_ms'       : t_total,
    }

    if verbose:
        print(f"  Chunks added    : {stats['chunks_added']}")
        print(f"  Embed calls     : {stats['embed_calls']}")
        print(f"  Vectors total   : {n_vec}")
        print(f"  Time: delete={t_del:.0f}ms  embed={t_emb:.0f}ms  total={t_total:.0f}ms")

    return stats


print("incremental_ingest() defined.")


## Step 9 — RAG Query

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant for a medical data visualization document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def rag_query(question: str, doc_id_filter: Optional[str] = None) -> Dict:
    t0   = time.time()
    qvec = embed_text(question)
    t_emb = (time.time() - t0) * 1000

    qfilter = None
    if doc_id_filter:
        qfilter = Filter(must=[FieldCondition(
            key="doc_id", match=MatchValue(value=doc_id_filter))])

    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec, limit=TOP_K,
        with_payload=True, query_filter=qfilter)

    hits    = [{'text': p.payload['text'],
                'page': p.payload.get('page','?'),
                'score': p.score} for p in resp.points]
    context = '\n\n'.join(
        f"[{i+1}] page={h['page']} score={h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(hits))
    answer  = call_llm(SYSTEM_PROMPT,
                       f"Context:\n{context}\n\nQuestion: {question}")
    t_total = (time.time() - t0) * 1000

    return {'question': question, 'answer': answer,
            'hits': hits, 'embed_ms': t_emb, 'total_ms': t_total}


print("rag_query() defined.")


## Step 10 — Demo Part 1: First Ingest (Full)

First run → manifest doesn't exist → all pages are treated as "added" → full ingest.


In [ ]:
print("=" * 65)
print("PART 1 — First ingest (cold start, no manifest)")
print("=" * 65)

# Clean any previous manifest
mp = manifest_path(DOC_ID)
if os.path.exists(mp):
    os.remove(mp)
    print("(Cleared previous manifest)")

stats_v1 = incremental_ingest(PDF_PATH, DOC_ID)

print()
print("Verifying RAG works after first ingest:")
r = rag_query("What are the best practices for choosing between a graph and a table?")
print(f"  Q: {r['question']}")
print(f"  A: {r['answer'][:300]}")


## Step 11 — Demo Part 2: Re-ingest Same Document

Edge case: same file ingested again. All hashes match → **zero embed calls, zero upsert operations**.


In [ ]:
print("=" * 65)
print("PART 2 — Re-ingest same document (idempotency check)")
print("=" * 65)

stats_same = incremental_ingest(PDF_PATH, DOC_ID)

print()
if stats_same['embed_calls'] == 0 and stats_same['chunks_deleted'] == 0:
    print("PASS: Zero embed calls, zero deletions — fully idempotent.")
else:
    print(f"UNEXPECTED: embed_calls={stats_same['embed_calls']}  deletions={stats_same['chunks_deleted']}")


## Step 12 — Demo Part 3: Simulated Document Updates

We simulate four real-world change scenarios using short synthetic PDFs.
Each page is a single ASCII sentence so `insert_text` → `get_text` roundtrips
exactly, giving reliable page-hash comparisons.

| Scenario | Simulation |
|---|---|
| **Page changed** | Modify page 3 text → expect old page-3 chunk deleted, 1 new chunk embedded |
| **Page added** | Add page 11 → expect only that page embedded |
| **Page deleted** | Remove page 11 → expect page-11 chunk deleted, zero embeds |
| **Multiple changes** | Pages 1 + 5 changed simultaneously |


In [ ]:
import io

# Short ASCII-only single-line pages that survive insert_text → get_text roundtrip exactly
SYNTHETIC_BASE = {
    1:  "Cover: Data Visualization Best Practices Guide version 2023 for healthcare analytics teams.",
    2:  "Section 1: Introduction to effective data communication in clinical settings.",
    3:  "Section 2: Chart selection criteria for dashboard design and reporting.",
    4:  "Section 3: Color theory and accessibility in medical data visualization.",
    5:  "Section 4: Tables vs graphs decision framework for data analysts.",
    6:  "Section 5: Interactive visualization tools and user experience design.",
    7:  "Section 6: Statistical charts and their appropriate use cases in medicine.",
    8:  "Section 7: Dashboard layout principles and information hierarchy design.",
    9:  "Section 8: Metrics and KPIs visualization for Medicaid programs.",
    10: "Section 9: Implementation examples and case studies from five states.",
}
SYNTHETIC_DOC_ID = "synthetic_doc"


def make_pdf_from_dict(pages_dict: Dict[int, str]) -> bytes:
    """Build a minimal synthetic PDF from {page_num: text} dict."""
    doc = fitz.open()
    for p in sorted(pages_dict):
        page = doc.new_page(width=595, height=842)
        page.insert_text((72, 72), pages_dict[p], fontsize=11)
    buf = io.BytesIO()
    doc.save(buf)
    doc.close()
    return buf.getvalue()


def save_pdf_dict(pages_dict: Dict[int, str], path: str) -> str:
    with open(path, 'wb') as f:
        f.write(make_pdf_from_dict(pages_dict))
    return path


# Clear any previous synthetic manifest
mp_syn = manifest_path(SYNTHETIC_DOC_ID)
if os.path.exists(mp_syn):
    os.remove(mp_syn)

v1_path = os.path.join(MANIFEST_DIR, "synthetic_v1.pdf")
save_pdf_dict(SYNTHETIC_BASE, v1_path)
print(f"Synthetic base document: {len(SYNTHETIC_BASE)} pages")
stats_syn_v1 = incremental_ingest(v1_path, SYNTHETIC_DOC_ID)


In [ ]:
# ── Scenario A: Page 3 changed ───────────────────────────────────────────────
print("=" * 65)
print("SCENARIO A — Single page changed (page 3)")
print("=" * 65)

v2_pages = dict(SYNTHETIC_BASE)
v2_pages[3] = "Section 2 UPDATED 2024: Scatter plot guidelines for correlation analysis in clinical dashboards."
v2_path = os.path.join(MANIFEST_DIR, "synthetic_v2.pdf")
save_pdf_dict(v2_pages, v2_path)

stats_a = incremental_ingest(v2_path, SYNTHETIC_DOC_ID)
print()
assert stats_a['pages_changed'] == 1, f"Expected 1 changed, got {stats_a['pages_changed']}"
assert stats_a['pages_unchanged'] == len(SYNTHETIC_BASE) - 1
print(f"PASS: 1 page changed, {stats_a['pages_unchanged']} unchanged, {stats_a['embed_calls']} embed calls")


In [ ]:
# ── Scenario B: New page added ───────────────────────────────────────────────
print("=" * 65)
print("SCENARIO B — New page added (page 11)")
print("=" * 65)

v3_pages = dict(v2_pages)
v3_pages[11] = "Appendix A: Extended case studies on Medicaid dashboard design from five state implementations."
v3_path = os.path.join(MANIFEST_DIR, "synthetic_v3.pdf")
save_pdf_dict(v3_pages, v3_path)

stats_b = incremental_ingest(v3_path, SYNTHETIC_DOC_ID)
print()
assert stats_b['pages_added'] == 1, f"Expected 1 added, got {stats_b['pages_added']}"
assert stats_b['pages_unchanged'] == len(v2_pages)
print(f"PASS: 1 page added, {stats_b['pages_unchanged']} unchanged, {stats_b['embed_calls']} embed calls")


In [ ]:
# ── Scenario C: Last page deleted (page 11) — zero embed calls ───────────────
print("=" * 65)
print("SCENARIO C — Last page deleted (page 11 removed)")
print("=" * 65)

v4_pages = {p: t for p, t in v3_pages.items() if p != 11}
v4_path  = os.path.join(MANIFEST_DIR, "synthetic_v4.pdf")
save_pdf_dict(v4_pages, v4_path)

stats_c = incremental_ingest(v4_path, SYNTHETIC_DOC_ID)
print()
assert stats_c['pages_deleted'] >= 1, f"Expected >=1 deleted, got {stats_c['pages_deleted']}"
assert stats_c['chunks_deleted'] > 0
assert stats_c['embed_calls'] == 0, f"Expected 0 embeds for pure deletion, got {stats_c['embed_calls']}"
print(f"PASS: {stats_c['pages_deleted']} page(s) deleted, {stats_c['chunks_deleted']} vectors removed, 0 embed calls")


In [ ]:
# ── Scenario D: Multiple pages changed simultaneously ─────────────────────────
print("=" * 65)
print("SCENARIO D — Multiple pages changed (pages 1 + 5)")
print("=" * 65)

v5_pages = dict(v4_pages)
v5_pages[1] = "Cover REVISED 2024: Data Visualization Best Practices for healthcare analytics teams with AI-assisted chart selection."
v5_pages[5] = "Section 4 REVISED: Distribution graphs now recommend violin plots alongside histograms for clinical outcome distributions."
v5_path = os.path.join(MANIFEST_DIR, "synthetic_v5.pdf")
save_pdf_dict(v5_pages, v5_path)

stats_d = incremental_ingest(v5_path, SYNTHETIC_DOC_ID)
print()
assert stats_d['pages_changed'] == 2, f"Expected 2 changed, got {stats_d['pages_changed']}"
print(f"PASS: {stats_d['pages_changed']} pages changed, "
      f"{stats_d['pages_unchanged']} unchanged, "
      f"{stats_d['embed_calls']} embed calls")


## Step 13 — Demo Part 4: RAG on Updated Index

Query the index after all incremental updates. The answers should reflect the latest state.


In [ ]:
print("=" * 65)
print("PART 4 — RAG queries on incrementally updated index")
print("=" * 65)
print()

questions = [
    "What does the 2024 edition cover for data visualization best practices?",
    "What are the updated guidelines for distribution graphs?",
    "What is covered in the Appendix A case studies?",
]

for q in questions:
    r = rag_query(q, doc_id_filter=SYNTHETIC_DOC_ID)
    safe_ans = r['answer'].encode('ascii', 'replace').decode()
    print(f"Q: {q}")
    print(f"   top hit: p{r['hits'][0]['page']} score={r['hits'][0]['score']:.3f}")
    print(f"A: {safe_ans[:280]}")
    print()


## Step 14 — Efficiency Report

In [ ]:
print("=" * 65)
print("INCREMENTAL RAG — Efficiency Report")
print("=" * 65)
print()

all_stats = [
    ("medicaid v1 — Full ingest (10p)",      stats_v1),
    ("medicaid v1 — Re-ingest (idempotent)", stats_same),
    ("synthetic v1 — Full ingest (10p)",     stats_syn_v1),
    ("v2: page 3 changed",                   stats_a),
    ("v3: page 11 added",                    stats_b),
    ("v4: page 11 deleted",                  stats_c),
    ("v5: pages 1+5 changed",                stats_d),
]

print(f"{'Scenario':<42} {'Embeds':>6} {'Deleted':>8} {'Saved%':>7} {'ms':>8}")
print("-" * 75)
for label, s in all_stats:
    full_embed = max(s['pages_total'], 1)
    saved_pct  = (1 - s['embed_calls'] / max(full_embed, 1)) * 100
    print(f"{label:<42} {s['embed_calls']:>6} {s['chunks_deleted']:>8} "
          f"{saved_pct:>6.0f}% {s['total_ms']:>7.0f}ms")

print()
print("--- medicaid doc ---")
manifest = load_manifest(DOC_ID)
print(f"  Pages tracked   : {len(manifest['pages'])}")
print(f"  Config hash     : {manifest['config_hash']}")
print()
print("--- synthetic doc ---")
manifest_syn = load_manifest(SYNTHETIC_DOC_ID)
print(f"  Pages tracked   : {len(manifest_syn['pages'])}")
print(f"  Vectors in index: {qdrant.get_collection(COLLECTION_NAME).points_count}")
print()
print("Notebook 31 — Incremental RAG complete.")


## Summary

### How it works

```
chunk_id = SHA-256(doc_id + "::" + page + "::" + text)
           ↑ used as Qdrant point ID → upsert is idempotent
```

### Manifest sidecar
```json
{
  "doc_id": "medicaid",
  "config_hash": "a1b2c3d4",
  "pages": {
    "1": {"page_hash": "abc123", "chunk_ids": ["id1", "id2"]}
  }
}
```

### Edge cases handled

| Scenario | Outcome |
|---|---|
| First ingest | Full embed, manifest created |
| Re-ingest same doc | Zero embed calls, zero deletes |
| Page content changed | Delete old chunk IDs, embed new chunks |
| Page added | Embed new page only |
| Page deleted | Delete chunk IDs, zero embed calls |
| Multiple changes | Diff detects all, processes only affected pages |
| Config params changed | config_hash mismatch → auto full re-index |
| Manifest missing | Treated as first ingest |

### API cost comparison (10-page doc, 3 chunks/page avg)

| Operation | Full re-index | Incremental (1 page changed) |
|---|---|---|
| Embed calls | 30 | 3 |
| Cost saving | — | ~90% |

> **Next: [32 — Multi-Tenant RAG](32_Multi_Tenant_RAG.ipynb)**
